In [1]:
# === IMPORT STANDARD & LSTM ===
import os, random, json, sys, time
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# === FIXED SEED + DISABLE GPU ===
import tensorflow as tf
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)
os.environ['PYTHONHASHSEED'] = str(seed_value)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
tf.config.set_visible_devices([], 'GPU')
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
print("⚠️ Deterministic mode active - CPU only")


sys.path.append(os.path.abspath("selenium-twitter-scraper/scraper"))
from twitter_scraper import Twitter_Scraper

# === KONFIGURASI ===
ticker = "AAPL"
base_path = "saved_sentiment_scraping"
sentiment_path = os.path.join(base_path, f"sentiment_{ticker.lower()}.csv")
os.makedirs(base_path, exist_ok=True)

# === SETUP LOGIN & SCRAPER ===
mail = "sitorocky@gmail.com"
username = "sito1405"
password = "Rocky1812@"
scraper = Twitter_Scraper(mail=mail, username=username, password=password, headlessState='yes', max_tweets=100)
scraper.login()

analyzer = SentimentIntensityAnalyzer()

# === KELOMPOK KATA KUNCI SENTIMEN ===
positive_keywords = ["invest", "dividend", "earnings", "bullish", "growth", "upgrade", "target price", "innovation", "profit", "services"]
negative_keywords = ["bearish", "downgrade", "loss", "drop", "decline", "risk", "recession", "crash", "inflation", "warning", "cut"]
neutral_keywords  = ["stock", "share", "price", "market", "valuation", "financial", "results", "shareholder", "outlook", "guidance", "revenue",
                     "aapl", "apple", "iphone", "mac", "ipad", "watch", "ecosystem", "tim cook", "ceo", "analyst", "rating"]
all_keywords = positive_keywords + negative_keywords + neutral_keywords

# === LOAD DATA SEBELUMNYA JIKA ADA ===
if os.path.exists(sentiment_path):
    existing_df = pd.read_csv(sentiment_path, parse_dates=['Date'], index_col='Date')
    if not existing_df.empty:
        last_date = existing_df.index[-1]
        start_date = last_date + timedelta(days=1)
        print(f"🔁 Melanjutkan dari tanggal terakhir: {start_date.strftime('%Y-%m-%d')}")
        results = existing_df['Sentiment'].to_dict()
    else:
        start_date = datetime(2017, 4, 9)
        results = {}
else:
    start_date = datetime(2017, 4, 9)
    results = {}

# === BATAS TANGGAL SCRAPING ===
end_date = datetime(2025, 9, 15)

# === FUNGSI SCRAPING DENGAN RETRY ===
def get_sentiment_for_date(date, max_retries=3):
    query = f"{ticker} stock since:{date.strftime('%Y-%m-%d')} until:{(date + timedelta(days=1)).strftime('%Y-%m-%d')}"
    print(f"🔄 Scraping tweets for {date.strftime('%Y-%m-%d')}...")

    for attempt in range(max_retries):
        try:
            scraper.scrape_tweets(
                max_tweets=100,
                scrape_query=query,
                scrape_latest=True,
                scrape_poster_details=False
            )
            tweets = scraper.get_tweets()
            scores = []
            filtered_count = 0

            for t in tweets:
                try:
                    text = t[4].lower()
                    count = sum(1 for kw in all_keywords if kw in text)
                    score = analyzer.polarity_scores(text)['compound']
                    if count >= 1 or abs(score) > 0.1:
                        scores.append(score)
                        filtered_count += 1
                except Exception as e:
                    print(f"⚠️ Error processing tweet: {e}")
                    continue

            if not scores and tweets:
                for t in tweets:
                    try:
                        text = t[4]
                        score = analyzer.polarity_scores(text)['compound']
                        scores.append(score)
                    except:
                        continue

            avg_sentiment = np.mean(scores) if scores else 0
            print(f"✅ {date.strftime('%Y-%m-%d')}: Avg sentiment = {avg_sentiment:.4f} (Tweets: {len(scores)}, Filtered: {filtered_count})")
            return avg_sentiment

        except Exception as e:
            print(f"⚠️ Attempt {attempt+1} failed: {e}")
            time.sleep(5)  # tunggu sebelum retry

    print(f"❌ Gagal scraping {date.strftime('%Y-%m-%d')} setelah {max_retries} percobaan.")
    return 0

# === MULAI SCRAPING ===
print("\n" + "="*50)
print("🚀 Memulai scraping Twitter")
print(f"📅 Periode: {start_date.strftime('%Y-%m-%d')} hingga {end_date.strftime('%Y-%m-%d')}")
print("="*50 + "\n")

current_date = start_date
save_interval = 7  # autosave setiap 7 hari
days_since_last_save = 0

while current_date <= end_date:
    sentiment = get_sentiment_for_date(current_date)
    results[current_date] = sentiment
    current_date += timedelta(days=1)
    days_since_last_save += 1

    # === AUTOSAVE PER 7 HARI ===
    if days_since_last_save >= save_interval:
        sentiment_df = pd.DataFrame.from_dict(results, orient='index', columns=['Sentiment'])
        sentiment_df.index.name = 'Date'
        sentiment_df.sort_index(inplace=True)
        sentiment_df.to_csv(sentiment_path)
        print(f"💾 Autosaved up to {sentiment_df.index[-1].strftime('%Y-%m-%d')}")
        days_since_last_save = 0

# === FINAL SAVE ===
sentiment_df = pd.DataFrame.from_dict(results, orient='index', columns=['Sentiment'])
sentiment_df.index.name = 'Date'
sentiment_df.sort_index(inplace=True)
sentiment_df.to_csv(sentiment_path)

print("\n" + "="*50)
print(f"✅ Sentimen selesai & disimpan di {sentiment_path}")
print("="*50)

scraper.driver.quit()


⚠️ Deterministic mode active - CPU only
Initializing Twitter Scraper...
Setup WebDriver...
Initializing FirefoxDriver...
WebDriver Setup Complete

🔐 Logging in to Twitter...
✅ Tidak ada verifikasi tambahan
✅ Login berhasil!

🔁 Melanjutkan dari tanggal terakhir: 2025-08-06

🚀 Memulai scraping Twitter
📅 Periode: 2025-08-06 hingga 2025-09-15

🔄 Scraping tweets for 2025-08-06...
Scraping Latest Tweets from AAPL stock since:2025-08-06 until:2025-08-07 search...
Progress: [[========================================]] 100.00% 100 of 100                                                  
Scraping Complete
Tweets: 100 out of 100

✅ 2025-08-06: Avg sentiment = 0.1986 (Tweets: 100, Filtered: 100)
🔄 Scraping tweets for 2025-08-07...
Scraping Latest Tweets from AAPL stock since:2025-08-07 until:2025-08-08 search...
Progress: [[========================================]] 100.00% 100 of 100                                                  
Scraping Complete
Tweets: 100 out of 100

✅ 2025-08-07: Avg sent